# System 2 Attention
0531 用 s2a paper 的prompt實作  prompt: 給定以下新聞，提取無偏見的部分，而不是他們的意見，這樣使用該文章將是為新聞的問題部分提供無偏見答案的良好內容。

0601 更改s2a prompt 讓他用更多的偏見來實作 prompt: 給定以下新聞，提取偏見的部分，保留他們的意見，這樣使用該文章將是為新聞的問題部分提供更多偏見的內容。

0604 更改 prompt 讓只保留偏見來實作 prompt: 請在以下新聞中，提取偏見的部分，並保留他們的意見，回傳該文章中含有偏見的內容。

In [1]:
# %pip install pandas
# %pip install numpy
# %pip install langchain_google_genai

In [2]:
import os
import textwrap
from datetime import datetime
import numpy_financial as npf
import math
import pandas as pd
import numpy as np
import json

from langchain_google_genai import ChatGoogleGenerativeAI
from google.generativeai.types import HarmBlockThreshold
from google.ai.generativelanguage_v1 import HarmCategory
from langchain_google_genai import ChatGoogleGenerativeAI

model = 'gemini-1.5-flash'
temperature = 1
top_p = 0.9
seed = 0
prompt_s2a = ""

llm = ChatGoogleGenerativeAI(
                google_api_key = os.getenv('GEMINI_API_KEY'),
                model=model,
                temperature=temperature,
                top_p=top_p,
                seed=seed,
                safety_settings={
                        HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: HarmBlockThreshold.BLOCK_NONE,
                        HarmCategory.HARM_CATEGORY_HARASSMENT: HarmBlockThreshold.BLOCK_NONE,
                        HarmCategory.HARM_CATEGORY_HATE_SPEECH: HarmBlockThreshold.BLOCK_NONE,
                        HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT : HarmBlockThreshold.BLOCK_NONE,
            })


/Users/yanyifu/Documents/_Coding/LLM Predict Stock/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [3]:
def llm_response(llm, text):
    ''' 
        filepath: string 檔案路徑
        text: string 問題指令
        
        return: string 回答
    '''

    try:
        response = llm.invoke(text)
        return response.content
    except Exception as e:
        error = "error: " + str(e)
        print(error)
        pass
    
    return error

In [4]:
# 轉換回答變成訊號
def response_to_signal(text):
    '''
        text: string 回答訊息
        
        return: int 回答訊號
    '''
    if text.find('no') > 0:
        signal = -1
    elif text.find('yes') > 0:
        signal = 1
    else:
        signal = 0

    return signal

# 設定要測試的股票

In [5]:
# 上市公司股票代號
# 代號 ['2618', '2881', '2880', '2211', '2731', '2707', '3023', '9941', '2308', '1514', '1101', '2542', '3036', '2385', '1303', '2912', '2323', '2603', '1419', '1604', '2486', '5269', '1314', '8033', '3017', '2903', '9943', '2368', '9904', '1102', '3231', '3004', '3035', '6176', '1103', '2609', '2002']
# 對應名稱 ['長榮航', '富邦金', '華南金', '長榮鋼', '雄獅', '晶華', '信邦', '裕融', '台達電', '亞力', '台泥', '興富發', '文曄', '群光', '南亞', '統一超', '中環', '長榮', '新光金', '聲寶', '一詮', '祥碩', '中石化', '遠雄', '奇鋐', '遠百', '好樂迪', '金像電', '寶成', '亞泥', '緯創', '豐達科', '智原', '瑞儀', '嘉泥', '陽明', '中鋼']
from datetime import datetime

start_time = datetime(2023, 1, 1)
end_time = datetime(2023, 12, 31)

stock_ids = ['2880', '2211', '2707']
stock_names = ['華南金', '長榮鋼', '晶華']


# stock_ids = ['2618', '2881', '2880', '2211', '2707']
# stock_names = ['長榮航', '富邦金', '華南金', '長榮鋼', '晶華']

# stock_ids = ['9941', '2308', '1514', '1101', '2542']
# stock_names = ['裕融', '台達電', '亞力', '台泥', '興富發']

In [9]:
count = 0
for idx, stock_id in enumerate(stock_ids):

    folder_path = os.path.dirname(os.path.abspath(os.getcwd())) + "/stock_news/" + stock_id + "/" + stock_id + "_news/"
    sorted_news_files = sorted(os.listdir(folder_path))

    # 載入價格資料
    df_price = pd.read_csv(os.path.dirname(os.path.abspath(os.getcwd())) + "/price_history/" + stock_id + "twse.csv", dtype=str)
    df_price.set_index('Date', inplace=True)  # 將'Date'行設置為索引
    
    acc_ = 0
    print(f'正在執行{stock_id}')
    # qurey = f"請找出對{stock_id}{stock_names[idx]}的股價有影響的新聞資料"
    signal = []
    counter = 0
    for date in df_price.index:
        # 檢查日期是否在參考日期之後
        try:
            date_obj = datetime.strptime(date, '%Y%m%d')
        except:
            continue

        if not (start_time <= date_obj <= end_time):
            continue
        
        news = ''
        sig = 0
        for index, file_name in enumerate(sorted_news_files):
            if file_name == date:
                
                with open(folder_path + "20230112", 'r', encoding='utf-8') as f:
                    news = f.read()
                print(f'{date}: {news}')
                res_s2a = llm_response(llm, f"""<s> [INST]給定以下新聞，提取偏見的部分，保留他們的意見，這樣使用該文章將是為新聞的問題部分提供更多偏見的內容。[/INST] </s> 
                                    [INST] 新聞: {news} 回答: [/INST]  """)
                print(res_s2a)


                count += 2
                break
        break
            

print(count)

正在執行2880
20230103: 台股今 (12) 日走勢震盪，早盤一度站回 14800 點，但多頭陷入觀望，電子股普遍壓回，航運股湧現賣壓，大盤回測 14700 點，加權指數終場下跌 19.8 點，收在 14731.64 點，成交量能急凍，僅不到 1500 億元。三大法人合計買超 25.29 億元，投信已連 5 買。觀察三大法人籌碼動向，外資今天小幅買超 0.01 億元；投信買超 18.22 億元，為連 5 買；自營商也買超 7.06 億元。三大法人合計買超 25.29 億元。晶圓代工龍頭台積電法說即將登場，市場屏息以待，今天股價表現有撐，終場上漲 2 元，收在 486.5 元；大立光同日舉行法說會，但受三實科技投資案喊停、外資降評等影響，今天股價續挫逾 1%；台達電、研華、華碩、聯詠、矽力 * - KY 等電子股賣壓也偏重。貨櫃三雄午盤賣壓湧現，長榮終場下跌逾 2%，失守季線，萬海終場也下跌 2.63%，其他航運股方面，裕民與英美資源集團簽署 4 艘 LNG 雙燃料動力散裝貨輪十年長約，激勵股價漲近 3%，航空雙雄拉回，華航收黑、長榮航平盤作收。金融股方面，富邦金今天股價逆勢上漲超過 1%，國泰金、開發金、元大金也守在盤上，但華南金、第一金下跌逾 1%。觀察盤面焦點個股，亞德客 - KY 去年營收創新高，獲外資看好，調升目標價，今天股價延續強勢，漲逾 4%，重回千金股行列。台泥近期在庫藏股、營收報捷、布局再生能源事業等題材激勵，股價表現強勢，今天續漲 2%，收復半年線。行政院會今天通過「疫後強化經濟與社會韌性及全民共享經濟成果特別條例」草案，作為全民普發 6000 元現金之法源，觀光類股受到買盤青睞，六角股價勁揚逾 7%，晶華、美食 - KY、聯發國際上漲逾 3%，八方雲集、亞洲藏壽司、洛碁也表現不俗。台股今 (12) 日走勢上沖下洗，多頭陷入觀望，靜待美國 CPI 報告出爐，加上台積電、大立光法說下午登場，整體交投偏淡，金融、傳產漲跌互見，航運股賣壓較重，加權指數終場下跌 19.8 點或 0.13%，收在 14731.64 點，回測 5 日線，成交值萎縮至 1474.6 億元。櫃買指數終場下跌 1.67 點或 0.89%，收在 186.85 點，成交值 507.89 億元。上市櫃成交值合計 1982.49 億元。晶圓代工龍頭台積電法說即將登場，市場屏息以待

KeyboardInterrupt: 

In [ ]:
count = 0
for idx, stock_id in enumerate(stock_ids):

    folder_path = os.path.dirname(os.path.abspath(os.getcwd())) + "/stock_news/" + stock_id + "/" + stock_id + "_news/"
    sorted_news_files = sorted(os.listdir(folder_path))

    # 載入價格資料
    df_price = pd.read_csv(os.path.dirname(os.path.abspath(os.getcwd())) + "/price_history/" + stock_id + "twse.csv", dtype=str)
    df_price.set_index('Date', inplace=True)  # 將'Date'行設置為索引
    
    acc_ = 0
    print(f'正在執行{stock_id}')
    # qurey = f"請找出對{stock_id}{stock_names[idx]}的股價有影響的新聞資料"
    signal = []
    counter = 0
    for date in df_price.index:
        # 檢查日期是否在參考日期之後
        try:
            date_obj = datetime.strptime(date, '%Y%m%d')
        except:
            continue

        if not (start_time <= date_obj <= end_time):
            continue
        
        news = ''
        sig = 0
        for index, file_name in enumerate(sorted_news_files):
            if file_name == date:
                
                with open(folder_path + file_name, 'r', encoding='utf-8') as f:
                    news = f.read()

                res_s2a = llm_response(llm, f"""<s> [INST] 請在以下新聞中，提取偏見的部分，並保留他們的意見，回傳該文章中含有偏見的內容。[/INST] </s> 
                                    [INST] 新聞: {news} 回答: [/INST]  """)
                
                res = llm_response(llm, f"""<s> [INST] 以下是今天的新聞資訊，請對於明天開盤時買進收盤時賣出的獲利可能性進行判斷，如果可以獲利請只回答#yes，否則只回答#no，如果與股市無關則回答#unknown，請盡量不要回答 #unknown[/INST] </s> 
                                    [INST] {res_s2a} 回答: [/INST]  """ ) 
                
                sig = response_to_signal(res)   

                count += 2
                break
            
        signal.append([date, sig]) # [日期, 訊號]
        # print(f'{date}: {sig}')

    # 輸出結果 json file
    output_data = {
        "date_start": start_time.strftime("%Y%m%d"),
        "date_end": end_time.strftime("%Y%m%d"),
        "model": model,
        "temperature": temperature,
        "top_p": top_p,
        "seed": seed,
        "raw_data": signal
    }

    # 轉換成 JSON 格式
    # ========================================================================================
    folder_path = "eval_s2a/output0603_morebias/"
    # ========================================================================================
    json_content = json.dumps(output_data, ensure_ascii=False, indent=4)
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)
    with open(folder_path + stock_id + "detail.json" , "w", encoding="UTF-8") as f:
        f.write(json_content)
        print("寫入檔案成功")

print(count)

# 對照組

In [ ]:
for idx, stock_id in enumerate(stock_ids):

    folder_path = os.path.dirname(os.path.abspath(os.getcwd())) + "/stock_news/" + stock_id + "/" + stock_id + "_news/"
    sorted_news_files = sorted(os.listdir(folder_path))

    # 載入價格資料
    df_price = pd.read_csv(os.path.dirname(os.path.abspath(os.getcwd())) + "/price_history/" + stock_id + "twse.csv", dtype=str)
    df_price.set_index('Date', inplace=True)  # 將'Date'行設置為索引
    
    acc_ = 0
    print(f'正在執行{stock_id}')
    # qurey = f"請找出對{stock_id}{stock_names[idx]}的股價有影響的新聞資料"
    signal = []
    counter = 0
    for date in df_price.index:
        # 檢查日期是否在參考日期之後
        try:
            date_obj = datetime.strptime(date, '%Y%m%d')
        except:
            continue

        if not (start_time <= date_obj <= end_time):
            continue

        news = ''
        sig = 0
        for index, file_name in enumerate(sorted_news_files):
            if file_name == date:
                
                with open(folder_path + file_name, 'r', encoding='utf-8') as f:
                    news = f.read()
                
                res_s2a = llm_response(llm, f"""<s> [INST] 給定以下新聞，提取無偏見的部分，而不是他們的意見，這樣使用該文章將是為新聞的問題部分提供無偏見答案的良好內容。[/INST] </s> 
                                    [INST] 新聞: {news} 回答: [/INST]  """)

                res = llm_response(llm, f"""<s> [INST] 以下是今天的新聞資訊，請對於明天開盤時買進收盤時賣出的獲利可能性進行判斷，如果可以獲利請只回答#yes，否則只回答#no，如果與股市無關則回答#unknown，請盡量不要回答 #unknown[/INST] </s> 
                                    [INST] {res_s2a} 回答: [/INST]  """ ) 
                 
                sig = response_to_signal(res) 
                break
        
        signal.append([date, sig]) # [日期, 訊號]
        # print(f'{date}: {sig}')

    # 輸出結果 json file
    output_data = {
        "date_start": start_time.strftime("%Y%m%d"),
        "date_end": end_time.strftime("%Y%m%d"),
        "model": model,
        "temperature": temperature,
        "top_p": top_p,
        "seed": seed,
        "raw_data": signal
    }

    # 轉換成 JSON 格式
    # ========================================================================================
    folder_path = "eval_s2a/output0605/"
    # ========================================================================================
    json_content = json.dumps(output_data, ensure_ascii=False, indent=4)
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)
    with open(folder_path + stock_id + "detail.json" , "w", encoding="UTF-8") as f:
        f.write(json_content)
        print("寫入檔案成功")